In [7]:
import pandas as pd
import os
import collections
from helper_functions import (
    convert_samples_to_binary,
    get_column_data_types,
    add_id_column,
    organize_photos_in_folders
)

In [8]:
try:
    df = pd.read_excel("../../../Datasets/excel/2dSpoofingDataset/GPS_Data_Simplified_2D_Feature_Map.xlsx")
except FileNotFoundError:
    print("Error: The file '../../../Datasets/excel/2dSpoofingDataset/GPS_Data_Simplified_2D_Feature_Map.xlsx' was not found.")
    print("Please make sure the path is correct.")
    exit()

In [9]:
# Data preprocessing

# Clean up column names
df.columns = df.columns.str.replace('[\\[\\]<>]', '_', regex=True)

# Make sure the 'Output' column exists
if 'Output' not in df.columns:
    raise ValueError("The dataset does not contain an 'Output' column. Please check your data.")

# Add an 'Id' column for image naming, using add_id_column from helper_functions
classification_id = 'Output'
df = add_id_column(df, classification_id)

# Drop label and timestamp columns from features (ignore if not present)
drop_cols = [classification_id, 'Id', 'timestamp']
X = df.drop(drop_cols, axis=1, errors='ignore')
y = df[classification_id]

# Encode y if needed
if y.dtype == object or y.dtype.name == 'category':
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    y = le.fit_transform(y)
    df[classification_id] = y  # update in df for consistency

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Row count match: {X.shape[0] == y.shape[0]}")

X shape: (510530, 13)
y shape: (510530,)
Row count match: True


In [10]:
feature_types = get_column_data_types(X)
print("Feature types:", feature_types)
print("Column names:", list(X.columns)
      )
print(X.dtypes)

Feature types: [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Column names: ['PRN', 'DO', 'PD', 'RX', 'TOW', 'CP', 'EC', 'LC', 'PC', 'PIP', 'PQP', 'TCD', 'CN0']
PRN      int64
DO     float64
PD     float64
RX     float64
TOW    float64
CP     float64
EC     float64
LC     float64
PC     float64
PIP    float64
PQP    float64
TCD    float64
CN0    float64
dtype: object


In [11]:
# Image Generation

# Update this path to where you want images to be saved
image_directory = "image-results-spoofing"
os.makedirs(image_directory, exist_ok=True)

# Precision and binary encoding settings
precision = 64     # number of binary digits per feature
one = 128          # pixel value for '1'
zero = 0           # pixel value for '0'
n_jobs = -1        # use all available cores

# Convert dataset samples to binary images
convert_samples_to_binary(
    X,
    df["Id"],
    image_directory,
    precision,
    one,
    zero,
    n_jobs,
    feature_types
)

# Organize images into folders by class
organize_photos_in_folders(image_directory, y)

Time to convert 510530 samples to binary: 198.74351048469543 seconds. Seconds per sample = 0.00038928860299041276
0
{0: 0, 1: 0, 2: 0, 3: 0}


In [12]:
# Print class distribution
print(collections.Counter(y))

Counter({0: 397825, 2: 44232, 1: 36458, 3: 32015})
